#1. Dependencies & Imports

In [ ]:
pip install openai wikipedia pandas

In [ ]:
import os, json, time, re, asyncio
import pandas as pd
import wikipedia
from datetime import datetime
from openai import AsyncOpenAI
from google.colab import auth
from google.auth import default

# ── Google Sheets ──────────────────────────────────────────────────────────────
import gspread
from google.oauth2.service_account import Credentials


# 2. Configuration

In [ ]:
# API Key
OPENROUTER_API_KEY = #"Your_API_key"

# Client for Decomposing
openai_client = AsyncOpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

# AI Model
MODELS = {
    "openai/gpt-4o-mini":"openai/gpt-4o-mini",
    "gryphe/mythomax-l2-13b":"gryphe/mythomax-l2-13b",
    "meta-llama/llama-3-8b-instruct":"meta-llama/llama-3-8b-instruct",
    "microsoft/phi-4":"microsoft/phi-4"
}

# System Prompt : 2 Model
SYSTEM_PROMPTS = {
    "neutral": (
        "คุณคือผู้ช่วยตอบคำถามทั่วไป"
    ),
    "fact_checker": (
        "คุณคือสารานุกรมเคลื่อนที่ที่เชี่ยวชาญข้อมูลบุคคลไทย "
        "ให้ข้อมูลที่แม่นยำและตรวจสอบได้เท่านั้น "
        "หากไม่แน่ใจข้อมูลใดให้ระบุอย่างชัดเจนว่า 'ไม่แน่ใจ' "
        "ห้ามคาดเดาหรือแต่งข้อมูลโดยเด็ดขาด"
    ),
}

# Entities : 1.) Entertainment 2.) Sports 3.) Politics
ENTITIES = [
    # 3.) Politician
    {"name": "จอมพล ป.พิบูลสงคราม", "fame": "Politician", "wiki_th": "จอมพล ป.พิบูลสงคราม"},
    {"name": "ปรีดี พนมยงค์", "fame": "Politician", "wiki_th": "ปรีดี พนมยงค์"},
    {"name": "จอมพล สฤษดิ์ ธนะรัชต์", "fame": "Politician", "wiki_th": "จอมพล สฤษดิ์ ธนะรัชต์"},
    {"name": "จอมพล ถนอม กิตติขจร", "fame": "Politician", "wiki_th": "จอมพล ถนอม กิตติขจร"},
    {"name": "จอมพล ประภาส จารุเสถียร", "fame": "Politician", "wiki_th": "จอมพล ประภาส จารุเสถียร"},
    {"name": "ควง อภัยวงศ์", "fame": "Politician", "wiki_th": "ควง อภัยวงศ์"},
    {"name": "เสนีย์ ปราโมช", "fame": "Politician", "wiki_th": "เสนีย์ ปราโมช"},
    {"name": "คึกฤทธิ์ ปราโมช", "fame": "Politician", "wiki_th": "คึกฤทธิ์ ปราโมช"},
    {"name": "เปรม ติณสูลานนท์", "fame": "Politician", "wiki_th": "เปรม ติณสูลานนท์"},
    {"name": "ชาติชาย ชุณหะวัณ", "fame": "Politician", "wiki_th": "ชาติชาย ชุณหะวัณ"},
    {"name": "ธานินทร์ กรัยวิเชียร", "fame": "Politician", "wiki_th": "ธานินทร์ กรัยวิเชียร"},
    {"name": "เกรียงศักดิ์ ชมะนันทน์", "fame": "Politician", "wiki_th": "เกรียงศักดิ์ ชมะนันทน์"},
    {"name": "พลเอก ชาติชาย ชุณหะวัณ", "fame": "Politician", "wiki_th": "พลเอก ชาติชาย ชุณหะวัณ"},
    {"name": "บรรจงศักดิ์ วงศ์ชัย", "fame": "Politician", "wiki_th": "บรรจงศักดิ์ วงศ์ชัย"},
    {"name": "นิพนธ์ บุญญามณี", "fame": "Politician", "wiki_th": "นิพนธ์ บุญญามณี"},
    {"name": "อภิวันท์ วิริยะชัย", "fame": "Politician", "wiki_th": "อภิวันท์ วิริยะชัย"},
    {"name": "สุชาติ ตันเจริญ", "fame": "Politician", "wiki_th": "สุชาติ ตันเจริญ"}
]

# 3. Google Sheet Helper

In [ ]:
def get_gsheet_client():
    auth.authenticate_user()
    creds, _ = default()
    return gspread.authorize(creds)

def export_to_gsheet(df: pd.DataFrame, summary_df: pd.DataFrame, position_df: pd.DataFrame):
    print("\nExporting to New Google Sheet ...")
    try:
        gc = get_gsheet_client()

        # Make a new file into my Google Drive
        timestamp_str = datetime.now().strftime("%Y-%m-%d_%H-%M")
        sheet_name = f"Hallucination_Matrix_{timestamp_str}"
        sh = gc.create(sheet_name)

        print(f"Complete Making a New file Name: '{sheet_name}'")
    except Exception as e:
        print(f"❌ Cannot make Google Sheet: {e}")
        return

    # Putting DataFrame into selected worksheet
    def write_sheet(worksheet_title: str, data: pd.DataFrame):
        try:
            ws = sh.worksheet(worksheet_title)
            ws.clear()
        except gspread.WorksheetNotFound:
            if worksheet_title == "Raw Data" and "Sheet1" in [w.title for w in sh.worksheets()]:
                ws = sh.worksheet("Sheet1")
                ws.update_title(worksheet_title)
            else:
                ws = sh.add_worksheet(title=worksheet_title, rows=5000, cols=30)

        # Changing NaN
        clean = data.fillna("").astype(str)
        headers = clean.columns.tolist()
        values  = [headers] + clean.values.tolist()
        ws.update(values, value_input_option="USER_ENTERED")
        print(f"  ✅ สร้างหน้า '{worksheet_title}' — {len(data)} rows")

    write_sheet("Raw Data",        df)
    write_sheet("FActScore",       summary_df if not summary_df.empty else pd.DataFrame({"msg": ["N.A."]}))
    write_sheet("Position Effect", position_df if not position_df.empty else pd.DataFrame({"msg": ["N.A."]}))

    # Abstention Rate sheet
    abstention = (
        df.drop_duplicates(subset=["model", "system_prompt", "entity_name"])
        .groupby(["model", "system_prompt"])["generation_status"]
        .apply(lambda g: round((g == "abstained").mean(), 4))
        .reset_index(name="abstention_rate")
    )
    write_sheet("Abstention Rate", abstention)

    #  Showing your google sheet URL
    print(f"\n🔗 Open your Google Sheet in this link: {sh.url}")

# 4. Core Pipeline

In [ ]:
# 1.) Making Person's Bio
GENERATION_USER_PROMPT = (
    "เขียนชีวประวัติโดยละเอียดของ {name} เป็นภาษาไทย "
    "ครอบคลุม: วันเกิด, ภูมิหลังครอบครัว, การศึกษา, ผลงานสำคัญ, "
    "รางวัลที่ได้รับ และบทบาทในสังคม "
    "เขียนเป็นย่อหน้าต่อเนื่อง ความยาวประมาณ 150-200 คำ"
)

async def generate_bio(model_key: str, prompt_key: str, entity: dict) -> dict:
    """Round 1: Making Bio"""
    try:
        resp = await openai_client.chat.completions.create(
            model=MODELS[model_key],
            messages=[
                {"role": "system",  "content": SYSTEM_PROMPTS[prompt_key]},
                {"role": "user",    "content": GENERATION_USER_PROMPT.format(
                                        name=entity["name"])},
            ],
            temperature=0.3,
            max_tokens=400,
        )
        bio = resp.choices[0].message.content.strip()
        abstained = any(kw in bio for kw in [
            "ไม่ทราบ", "ไม่มีข้อมูล", "ไม่แน่ใจ",
            "ไม่พบข้อมูล", "ขออภัย", "I don't know"
        ])
        return {
            "status": "abstained" if abstained else "generated",
            "bio": bio,
            "tokens_used": resp.usage.total_tokens,
        }
    except Exception as e:
        return {"status": "error", "bio": "", "error": str(e), "tokens_used": 0}

# 2. Decomposing the Bio

DECOMPOSE_SYSTEM = (
    "คุณเป็นผู้ช่วยสกัดข้อเท็จจริง ตอบเป็น JSON Array เท่านั้น "
    "ห้ามมีข้อความอื่นนอกจาก JSON"
)

DECOMPOSE_USER = """จงแยกชีวประวัติต่อไปนี้ออกเป็นรายการ "ข้อเท็จจริงย่อย"
แต่ละข้อต้องเป็นประโยคเดียว สั้น ชัดเจน และตรวจสอบได้อิสระ

ชีวประวัติ:
{bio}

ตอบในรูปแบบ JSON Array เช่น:
["ชื่อ เกิดวันที่ X เดือน Y พ.ศ. Z", "ชื่อ สำเร็จการศึกษาจาก X", ...]

ข้อควรระวัง:
- แต่ละ item ต้องมีชื่อบุคคลเป็น subject ชัดเจน
- ห้ามรวมข้อเท็จจริง 2 อย่างในประโยคเดียว
- ตัดความคิดเห็นหรือการประเมินค่าออก เก็บแต่ข้อเท็จจริง"""

async def decompose_bio(bio: str) -> list:
    """Round 2: Decompose Bio into atomic facts"""
    if not bio:
        return []
    try:
        resp = await openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": DECOMPOSE_SYSTEM},
                {"role": "user",   "content": DECOMPOSE_USER.format(bio=bio)},
            ],
            temperature=0.0,
            max_tokens=600,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"```json|```", "", raw).strip()
        facts = json.loads(raw)
        return [str(f) for f in facts] if isinstance(facts, list) else []
    except Exception:
        # fallback: แยกด้วย newline
        return [line.strip() for line in bio.split("\n") if len(line.strip()) > 15]


# 3. Verifying in Wikipedia

wikipedia.set_lang("th")

def get_wikipedia_summary(wiki_title) -> str:
    """ดึง Wikipedia summary"""
    if not wiki_title:
        return ""
    try:
        page = wikipedia.page(wiki_title, auto_suggest=False)
        return page.content[:3000]
    except Exception:
        return ""

VERIFY_SYSTEM = (
    "คุณเป็นผู้ตรวจสอบข้อเท็จจริงผู้เชี่ยวชาญ ตอบเป็น JSON เท่านั้น "
    "ห้ามมีข้อความอื่นนอกจาก JSON"
)

VERIFY_USER = """ตรวจสอบข้อเท็จจริงต่อไปนี้โดยใช้ข้อมูลจาก Wikipedia ที่ให้มา

ข้อเท็จจริงที่ต้องตรวจสอบ: "{fact}"

ข้อมูล Wikipedia อ้างอิง:
{wiki_evidence}

ตอบในรูปแบบ JSON:
{{
  "label": "Supported" | "Not-Supported" | "Undecidable",
  "confidence": 0.0-1.0,
  "reason": "อธิบายสั้นๆ ว่าทำไม"
}}

หมายเหตุ:
- "Supported"     = Wikipedia ยืนยันข้อเท็จจริงนี้ชัดเจน
- "Not-Supported" = Wikipedia โต้แย้งหรือข้อมูลแตกต่างจากนี้
- "Undecidable"   = Wikipedia ไม่มีข้อมูลพอให้ตัดสิน"""

async def verify_fact(fact: str, wiki_evidence: str) -> dict:
    """Round 3: ตรวจสอบ atomic fact กับ Wikipedia"""
    if not wiki_evidence:
        return {
            "label":      "Undecidable",
            "confidence": 0.0,
            "reason":     "ไม่มีข้อมูล Wikipedia สำหรับตรวจสอบ",
        }
    try:
        resp = await openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": VERIFY_SYSTEM},
                {"role": "user",   "content": VERIFY_USER.format(
                    fact=fact, wiki_evidence=wiki_evidence)},
            ],
            temperature=0.0,
            max_tokens=150,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"```json|```", "", raw).strip()
        result = json.loads(raw)
        if result.get("label") not in ("Supported", "Not-Supported", "Undecidable"):
            result["label"] = "Undecidable"
        return result
    except Exception as e:
        return {"label": "Undecidable", "confidence": 0.0, "reason": f"Error: {e}"}


# 5. Main

In [ ]:
# ─────────────────────────────────────────────
#  MAIN PIPELINE
# ─────────────────────────────────────────────

async def process_one(model_key: str, prompt_key: str,
                      entity: dict, wiki_cache: dict) -> list:

    if not isinstance(entity, dict):
        return []

    """Run pipeline 3 times for 1 combination"""
    rows = []
    base = {
        "timestamp":         datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model":             model_key,
        "system_prompt":     prompt_key,
        "entity_name":       entity["name"],
        "fame_level":        entity["fame"],
    }

    # Round 1
    gen = await generate_bio(model_key, prompt_key, entity)
    base["generation_status"] = gen["status"]
    base["bio_text"]          = gen.get("bio", "")
    base["tokens_round1"]     = gen.get("tokens_used", 0)

    if gen["status"] in ("error", "abstained"):
        rows.append({**base,
                     "atomic_fact":   "",
                     "fact_position": None,
                     "fact_total":    None,
                     "label":         "N/A",
                     "confidence":    None,
                     "reason":        gen.get("error", "abstained")})
        return rows

    # Round 2
    facts = await decompose_bio(gen["bio"])

    # Wikipedia (cached)
    wiki_key = entity.get("wiki_th") or entity["name"]
    if wiki_key not in wiki_cache:
      result = get_wikipedia_summary(entity.get("wiki_th"))
      # Checking if result is string
      wiki_cache[wiki_key] = result if isinstance(result, str) else ""
    evidence = wiki_cache.get(wiki_key, "")

    # Round 3
    for idx, fact in enumerate(facts):
        result = await verify_fact(fact, evidence)
        rows.append({
            **base,
            "atomic_fact":   fact,
            "fact_position": idx,
            "fact_total":    len(facts),
            "label":         result["label"],
            "confidence":    result.get("confidence"),
            "reason":        result.get("reason", ""),
        })
        await asyncio.sleep(0.1)

    return rows


async def run_all(
    entities:         list  = ENTITIES,
    models:           list  = list(MODELS.keys()),
    prompts:          list  = list(SYSTEM_PROMPTS.keys()),
    output_csv:       str   = "hallucination_matrix_results.csv",
    checkpoint_every: int   = 50,
    export_gsheet:    bool  = True,
) -> pd.DataFrame:

    if isinstance(entities, str): entities = [entities]
    if isinstance(models, str):   models   = [models]
    if isinstance(prompts, str):  prompts  = [prompts]


    all_rows   = []
    wiki_cache = {}
    total  = len(entities) * len(models) * len(prompts)
    count  = 0

    print(f"{'='*55}")
    print(f" Hallucination Matrix — {total} combinations")
    print(f"{'='*55}\n")

    for entity in entities:
        for model_key in models:
            for prompt_key in prompts:
                count += 1
                print(f"[{count:>4}/{total}] {model_key} | {prompt_key:<12} | {entity['name']}")

                rows = await process_one(model_key, prompt_key, entity, wiki_cache)
                all_rows.extend(rows)

                if count % checkpoint_every == 0:
                    pd.DataFrame(all_rows).to_csv(output_csv, index=False, encoding="utf-8-sig")
                    print(f"  ✓ Checkpoint saved ({len(all_rows)} rows)")

                await asyncio.sleep(0.3)

    df = pd.DataFrame(all_rows)
    df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\n✅ Done! {len(df)} rows → {output_csv}")

    # Calculate summary before export
    summary_df  = compute_factscore(df)
    position_df = analyze_position_effect(df)

    if export_gsheet:
        export_to_gsheet(df, summary_df, position_df)

    return df

# Analysis

def compute_factscore(df: pd.DataFrame) -> pd.DataFrame:
    """FActScore = Supported / (Supported + Not-Supported)"""
    verifiable = df[df["label"].isin(["Supported", "Not-Supported"])].copy()
    if verifiable.empty:
        return pd.DataFrame()

    # ✅ FIX: ไม่ใช้ include_groups (deprecated ใน pandas >= 2.2)
    result = (
        verifiable
        .groupby(["model", "system_prompt", "fame_level"])
        .apply(lambda g: pd.Series({
            "factscore":          round((g["label"] == "Supported").mean(), 4),
            "hallucination_rate": round((g["label"] == "Not-Supported").mean(), 4),
            "n_facts":            len(g),
            "n_supported":        int((g["label"] == "Supported").sum()),
            "n_not_supported":    int((g["label"] == "Not-Supported").sum()),
        }))
        .reset_index()
    )
    return result


def analyze_position_effect(df: pd.DataFrame) -> pd.DataFrame:
    verifiable = df[
        df["label"].isin(["Supported", "Not-Supported"]) &
        df["fact_total"].notna() &
        df["fact_position"].notna()
    ].copy()

    verifiable = verifiable[verifiable["fact_total"] >= 3]
    if verifiable.empty:
        return pd.DataFrame()

    def position_group(row):
        pos = row["fact_position"] / max(row["fact_total"] - 1, 1)
        if pos < 0.33:   return "early"
        elif pos < 0.67: return "middle"
        else:            return "late"

    verifiable["position_group"] = verifiable.apply(position_group, axis=1)

    # ✅ FIX: ไม่ใช้ include_groups
    return (
        verifiable
        .groupby(["model", "system_prompt", "position_group"])
        .apply(lambda g: pd.Series({
            "hallucination_rate": round((g["label"] == "Not-Supported").mean(), 4),
            "n_facts":            len(g),
        }))
        .reset_index()
    )


def print_summary(df: pd.DataFrame) -> None:
    print("\n" + "="*55)
    print(" FACTSCORE SUMMARY")
    print("="*55)
    fs = compute_factscore(df)
    if not fs.empty:
        print(fs.to_string(index=False))

    print("\n" + "="*55)
    print(" ABSTENTION RATE")
    print("="*55)
    rate = (
        df.drop_duplicates(subset=["model", "system_prompt", "entity_name"])
        .groupby(["model", "system_prompt"])["generation_status"]
        .apply(lambda g: round((g == "abstained").mean(), 4))
        .reset_index(name="abstention_rate")
    )
    print(rate.to_string(index=False))

    print("\n" + "="*55)
    print(" POSITION EFFECT")
    print("="*55)
    pos = analyze_position_effect(df)
    if not pos.empty:
        print(pos.to_string(index=False))




# 6.Execution

In [ ]:
# ─────────────────────────────────────────────
#  ENTRY POINT
# ─────────────────────────────────────────────

async def main():
    df = await run_all(
        entities=ENTITIES,
        export_gsheet=True,   # ← ตั้งเป็น False ถ้ายังไม่ได้ตั้งค่า Google Sheet
    )
    print_summary(df)


if __name__ == "__main__":
    await main()

 Hallucination Matrix — 136 combinations

[   1/136] openai/gpt-4o-mini | neutral      | จอมพล ป.พิบูลสงคราม
[   2/136] openai/gpt-4o-mini | fact_checker | จอมพล ป.พิบูลสงคราม
[   3/136] gryphe/mythomax-l2-13b | neutral      | จอมพล ป.พิบูลสงคราม
[   4/136] gryphe/mythomax-l2-13b | fact_checker | จอมพล ป.พิบูลสงคราม
[   5/136] meta-llama/llama-3-8b-instruct | neutral      | จอมพล ป.พิบูลสงคราม
[   6/136] meta-llama/llama-3-8b-instruct | fact_checker | จอมพล ป.พิบูลสงคราม
[   7/136] microsoft/phi-4 | neutral      | จอมพล ป.พิบูลสงคราม
[   8/136] microsoft/phi-4 | fact_checker | จอมพล ป.พิบูลสงคราม
[   9/136] openai/gpt-4o-mini | neutral      | ปรีดี พนมยงค์
[  10/136] openai/gpt-4o-mini | fact_checker | ปรีดี พนมยงค์
[  11/136] gryphe/mythomax-l2-13b | neutral      | ปรีดี พนมยงค์
[  12/136] gryphe/mythomax-l2-13b | fact_checker | ปรีดี พนมยงค์
[  13/136] meta-llama/llama-3-8b-instruct | neutral      | ปรีดี พนมยงค์
[  14/136] meta-llama/llama-3-8b-instruct | fact_checker | ปรีดี พนมยงค

/tmp/ipykernel_30899/4217302643.py:128: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/tmp/ipykernel_30899/4217302643.py:163: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


Complete Making a New file Name: 'Hallucination_Matrix_2026-04-25_13-05'
  ✅ สร้างหน้า 'Raw Data' — 1406 rows
  ✅ สร้างหน้า 'FActScore' — 8 rows
  ✅ สร้างหน้า 'Position Effect' — 24 rows
  ✅ สร้างหน้า 'Abstention Rate' — 8 rows

🔗 Open your Google Sheet in this link: https://docs.google.com/spreadsheets/d/1_8aOn7G7CI7PVTLh0P_Tx5GYAPKCH4E8xMKf0fKVeaU

 FACTSCORE SUMMARY
                         model system_prompt fame_level  factscore  hallucination_rate  n_facts  n_supported  n_not_supported
        gryphe/mythomax-l2-13b  fact_checker Politician     0.2371              0.7629     97.0         23.0             74.0
        gryphe/mythomax-l2-13b       neutral Politician     0.2121              0.7879     99.0         21.0             78.0
meta-llama/llama-3-8b-instruct  fact_checker Politician     0.1974              0.8026    152.0         30.0            122.0
meta-llama/llama-3-8b-instruct       neutral Politician     0.1813              0.8187    193.0         35.0            158.

/tmp/ipykernel_30899/4217302643.py:128: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/tmp/ipykernel_30899/4217302643.py:163: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [ ]:
print(type(ENTITIES), ENTITIES[:1])
print(type(list(MODELS.keys())), list(MODELS.keys()))
print(type(list(SYSTEM_PROMPTS.keys())), list(SYSTEM_PROMPTS.keys()))

<class 'list'> [{'name': 'จอมพล ป.พิบูลสงคราม', 'fame': 'Politician', 'wiki_th': 'จอมพล ป.พิบูลสงคราม'}]
<class 'list'> ['openai/gpt-4o-mini', 'gryphe/mythomax-l2-13b', 'meta-llama/llama-3-8b-instruct', 'microsoft/phi-4']
<class 'list'> ['neutral', 'fact_checker']
